### AgenticAI Exercise

#### Keith R. Holdaway FGS

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import json

# =============================================
# 1. TOOLS: External systems the AI can interact with
# =============================================

class PetrophysicalTools:
    """Tools for petrophysical calculations and file operations"""
    
    @staticmethod
    def load_well_data(filepath):
        """Tool to load well log data from CSV"""
        try:
            df = pd.read_csv(filepath)
            print(f"✓ Loaded well data: {len(df)} rows")
            return df
        except Exception as e:
            return f"Error loading file: {e}"
    
    @staticmethod
    def save_to_csv(data, filename):
        """Tool to save results to CSV"""
        try:
            if isinstance(data, pd.DataFrame):
                data.to_csv(filename, index=False)
            else:
                pd.DataFrame(data).to_csv(filename, index=False)
            print(f"✓ Saved results to: {filename}")
            return f"File saved successfully: {filename}"
        except Exception as e:
            return f"Error saving file: {e}"
    
    @staticmethod
    def calculate_porosity(rhob, rhob_matrix=2.65, rhob_fluid=1.0):
        """Density porosity calculation"""
        porosity = (rhob_matrix - rhob) / (rhob_matrix - rhob_fluid)
        return np.clip(porosity, 0, 1)  # Clip to valid range
    
    @staticmethod
    def calculate_vshale(gr, gr_clean=20, gr_shale=120):
        """Gamma Ray shale volume calculation"""
        vshale = (gr - gr_clean) / (gr_shale - gr_clean)
        return np.clip(vshale, 0, 1)
    
    @staticmethod
    def calculate_sw_archie(rt, porosity, rw=0.05, a=1.0, m=2.0, n=2.0):
        """Archie water saturation calculation"""
        f = a / (porosity ** m)
        sw = (f * rw / rt) ** (1/n)
        return np.clip(sw, 0, 1)
    
    @staticmethod
    def identify_pay_zones(porosity, vshale, sw, 
                          porosity_cutoff=0.08, 
                          vshale_cutoff=0.5, 
                          sw_cutoff=0.6):
        """Identify potential pay zones based on cutoffs"""
        pay_flag = (porosity > porosity_cutoff) & (vshale < vshale_cutoff) & (sw < sw_cutoff)
        return pay_flag.astype(int)

# =============================================
# 2. LANGUAGE MODEL (Simulated): Decision engine
# =============================================

class PetrophysicalAI:
    """Simulated Language Model for petrophysical decision-making"""
    
    def __init__(self):
        self.expert_knowledge = {
            "workflow": [
                "load_data",
                "quality_check", 
                "calculate_porosity",
                "calculate_vshale",
                "calculate_sw",
                "identify_pay",
                "summarize_results",
                "save_output"
            ],
            "default_parameters": {
                "rhob_matrix": 2.65,
                "rhob_fluid": 1.0,
                "gr_clean": 20,
                "gr_shale": 120,
                "rw": 0.05,
                "a": 1.0,
                "m": 2.0,
                "n": 2.0
            },
            "cutoffs": {
                "porosity": 0.08,
                "vshale": 0.5,
                "sw": 0.6
            }
        }
    
    def plan_analysis(self, user_request):
        """LM decides the analysis plan based on user request"""
        print("🧠 AI Petrophysicist: Planning analysis workflow...")
        
        if "pay" in user_request.lower():
            plan = self.expert_knowledge["workflow"]
        elif "porosity" in user_request.lower():
            plan = self.expert_knowledge["workflow"][:4]  # Stop after porosity
        else:
            plan = self.expert_knowledge["workflow"][:3]  # Basic analysis
        
        return plan
    
    def decide_parameters(self, data_sample):
        """LM decides which parameters to use based on data characteristics"""
        params = self.expert_knowledge["default_parameters"].copy()
        
        # Simple heuristic: adjust parameters based on data range
        if 'GR' in data_sample.columns:
            gr_max = data_sample['GR'].max()
            if gr_max > 150:
                params['gr_shale'] = 180  # Adjust for high GR shale
        
        return params

# =============================================
# 3. ORCHESTRATION: Manages workflow and memory
# =============================================

class AnalysisOrchestrator:
    """Orchestrates the entire petrophysical analysis workflow"""
    
    def __init__(self):
        self.tools = PetrophysicalTools()
        self.lm = PetrophysicalAI()
        self.memory = {
            "current_step": None,
            "completed_steps": [],
            "data": None,
            "parameters": {},
            "results": {},
            "timestamp": datetime.now().isoformat()
        }
    
    def execute_workflow(self, user_request, input_file):
        """Main method to execute the complete analysis"""
        print(f"🚀 Starting analysis: {user_request}")
        print(f"📁 Input file: {input_file}")
        
        # Step 1: LM creates the plan
        plan = self.lm.plan_analysis(user_request)
        self.memory["plan"] = plan
        
        # Execute each step in the plan
        for step in plan:
            self.memory["current_step"] = step
            print(f"\n📋 Step: {step.replace('_', ' ').title()}")
            
            result = self.execute_step(step, input_file)
            self.memory["completed_steps"].append(step)
            self.memory["results"][step] = result
            
            if "error" in str(result).lower():
                print(f"❌ Error in step {step}: {result}")
                break
        
        # Final summary
        return self.generate_summary()
    
    def execute_step(self, step, input_file):
        """Execute a single step using Tools based on LM decisions"""
        try:
            if step == "load_data":
                data = self.tools.load_well_data(input_file)
                if isinstance(data, pd.DataFrame):
                    self.memory["data"] = data
                    self.memory["parameters"] = self.lm.decide_parameters(data.head())
                return data
            
            elif step == "quality_check":
                data = self.memory["data"]
                return f"Data quality: {len(data)} rows, {len(data.columns)} columns"
            
            elif step == "calculate_porosity":
                data = self.memory["data"]
                params = self.memory["parameters"]
                data['PHIT'] = self.tools.calculate_porosity(
                    data['RHOB'], params['rhob_matrix'], params['rhob_fluid']
                )
                return f"Porosity calculated: avg={data['PHIT'].mean():.3f}"
            
            elif step == "calculate_vshale":
                data = self.memory["data"]
                params = self.memory["parameters"]
                data['VSH'] = self.tools.calculate_vshale(
                    data['GR'], params['gr_clean'], params['gr_shale']
                )
                return f"VShale calculated: avg={data['VSH'].mean():.3f}"
            
            elif step == "calculate_sw":
                data = self.memory["data"]
                params = self.memory["parameters"]
                data['SW'] = self.tools.calculate_sw_archie(
                    data['RT'], data['PHIT'], params['rw'], params['a'], params['m'], params['n']
                )
                return f"Water saturation calculated: avg={data['SW'].mean():.3f}"
            
            elif step == "identify_pay":
                data = self.memory["data"]
                cutoffs = self.lm.expert_knowledge["cutoffs"]
                data['PAY_FLAG'] = self.tools.identify_pay_zones(
                    data['PHIT'], data['VSH'], data['SW'],
                    cutoffs['porosity'], cutoffs['vshale'], cutoffs['sw']
                )
                pay_count = data['PAY_FLAG'].sum()
                return f"Pay zones identified: {pay_count} intervals"
            
            elif step == "save_output":
                data = self.memory["data"]
                output_file = f"analysis_results_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
                return self.tools.save_to_csv(data, output_file)
            
            elif step == "summarize_results":
                return self.generate_detailed_summary()
                
        except Exception as e:
            return f"Error in {step}: {e}"
    
    def generate_summary(self):
        """Generate a final summary of the analysis"""
        data = self.memory["data"]
        
        summary = {
            "analysis_completed": datetime.now().isoformat(),
            "total_steps": len(self.memory["completed_steps"]),
            "data_points": len(data) if data is not None else 0,
            "parameters_used": self.memory["parameters"]
        }
        
        if data is not None:
            if 'PAY_FLAG' in data.columns:
                summary["pay_zones_identified"] = int(data['PAY_FLAG'].sum())
                summary["net_pay_thickness"] = f"{data['PAY_FLAG'].sum() * 0.5}m"  # Assuming 0.5m sampling
            
            if 'PHIT' in data.columns:
                summary["average_porosity"] = f"{data['PHIT'].mean():.3f}"
        
        return summary
    
    def generate_detailed_summary(self):
        """Generate detailed results summary"""
        data = self.memory["data"]
        if data is None:
            return "No data available for summary"
        
        summary_lines = ["\n" + "="*50]
        summary_lines.append("PETROPHYSICAL ANALYSIS SUMMARY")
        summary_lines.append("="*50)
        summary_lines.append(f"Total depth intervals: {len(data)}")
        
        if 'PHIT' in data.columns:
            summary_lines.append(f"Average Porosity (PHIT): {data['PHIT'].mean():.3f}")
            summary_lines.append(f"Max Porosity: {data['PHIT'].max():.3f}")
        
        if 'VSH' in data.columns:
            summary_lines.append(f"Average VShale: {data['VSH'].mean():.3f}")
        
        if 'SW' in data.columns:
            summary_lines.append(f"Average Water Saturation: {data['SW'].mean():.3f}")
        
        if 'PAY_FLAG' in data.columns:
            pay_count = data['PAY_FLAG'].sum()
            summary_lines.append(f"Pay Zones Identified: {pay_count}")
            summary_lines.append(f"Net Pay: {pay_count * 0.5:.1f} meters")
        
        summary_lines.append("="*50)
        return "\n".join(summary_lines)

# =============================================
# 4. SAMPLE DATA GENERATION AND MAIN PROGRAM
# =============================================

def generate_sample_well_data(filename="sample_well_data.csv"):
    """Generate sample well log data for testing"""
    depth = np.arange(2000, 2500, 0.5)  # 1000 data points, 0.5m spacing
    
    # Create realistic log responses
    gr_base = 20 + 80 * np.sin(depth/50)  # Cyclic GR pattern
    gr = gr_base + np.random.normal(0, 5, len(depth))
    
    # Porosity trend with some noise
    porosity_trend = 0.15 + 0.1 * np.sin(depth/100)
    porosity = np.clip(porosity_trend + np.random.normal(0, 0.02, len(depth)), 0.05, 0.25)
    
    # Calculate density from porosity
    rhob_matrix = 2.65
    rhob_fluid = 1.0
    rhob = rhob_matrix - porosity * (rhob_matrix - rhob_fluid)
    
    # Resistivity based on porosity and water saturation
    rw = 0.05
    a, m, n = 1.0, 2.0, 2.0
    sw = 0.2 + 0.6 * np.sin(depth/80)  # Varying water saturation
    rt = (a * rw) / (porosity ** m * sw ** n)
    
    # Add some noise to resistivity
    rt = rt + np.random.normal(0, 0.5, len(depth))
    
    # Create DataFrame
    data = pd.DataFrame({
        'DEPTH': depth,
        'GR': gr,
        'RHOB': rhob,
        'RT': rt
    })
    
    # Save to CSV
    data.to_csv(filename, index=False)
    print(f"✓ Generated sample well data: {filename}")
    return filename

def main():
    """Main function to demonstrate the Agentic AI system"""
    print("🤖 PETROPHYSICAL AI AGENT DEMONSTRATION")
    print("=" * 50)
    
    # Step 1: Generate sample data
    input_file = generate_sample_well_data()
    
    # Step 2: Initialize the orchestrator (our Agentic AI system)
    orchestrator = AnalysisOrchestrator()
    
    # Step 3: Execute different analysis requests
    requests = [
        "Analyze this well for potential pay zones",
        "Calculate basic petrophysical parameters",
        "Full petrophysical analysis with pay identification"
    ]
    
    for i, request in enumerate(requests, 1):
        print(f"\n{'#'*60}")
        print(f"ANALYSIS REQUEST {i}: {request}")
        print(f"{'#'*60}")
        
        results = orchestrator.execute_workflow(request, input_file)
        
        print("\n📊 RESULTS SUMMARY:")
        for key, value in results.items():
            print(f"  {key}: {value}")
        
        # Reset orchestrator for next analysis
        orchestrator = AnalysisOrchestrator()

if __name__ == "__main__":
    main()

🤖 PETROPHYSICAL AI AGENT DEMONSTRATION
✓ Generated sample well data: sample_well_data.csv

############################################################
ANALYSIS REQUEST 1: Analyze this well for potential pay zones
############################################################
🚀 Starting analysis: Analyze this well for potential pay zones
📁 Input file: sample_well_data.csv
🧠 AI Petrophysicist: Planning analysis workflow...

📋 Step: Load Data
✓ Loaded well data: 1000 rows

📋 Step: Quality Check

📋 Step: Calculate Porosity

📋 Step: Calculate Vshale

📋 Step: Calculate Sw

📋 Step: Identify Pay

📋 Step: Summarize Results

📋 Step: Save Output
✓ Saved results to: analysis_results_20251001_0928.csv

📊 RESULTS SUMMARY:
  analysis_completed: 2025-10-01T09:28:17.644771
  total_steps: 8
  data_points: 1000
  parameters_used: {'rhob_matrix': 2.65, 'rhob_fluid': 1.0, 'gr_clean': 20, 'gr_shale': 120, 'rw': 0.05, 'a': 1.0, 'm': 2.0, 'n': 2.0}
  pay_zones_identified: 325
  net_pay_thickness: 162.5m
  aver